# ResNet-50 (layer4[2] Unfreeze — WD 1e-3 + L1 + L2 Penalty)

## Changes from Iteration 05

| | 05. layer4 + L1+L2 | 06. layer4[2] + L1+L2 |
|---|---|---|
| Unfrozen | All of `layer4` (~15M params) | `layer4[2]` only (~4.7M params) |
| Weight decay | 1e-3 | 1e-3 |
| L1 lambda | 1e-4 | 1e-4 |
| L2 lambda | 1e-4 | 1e-4 |
| LR — unfrozen block | 1e-4 | 1e-4 |
| LR — FC head | 1e-3 | 1e-3 |
| Dropout | 0.5 | 0.5 |
| Epochs | 30 | 30 |

## Rationale

Iteration 05 confirmed that L1+L2 penalties meaningfully improve generalisation — test F2 jumped from 0.5859 (04) to **0.6320** (05), with AUC-ROC reaching **0.8864**. However the train/val gap is still ~0.20, meaning the model is still overfitting with 15M trainable params.

Iteration 03 showed that restricting to `layer4[2]` alone reduces the gap to ~0.09 — but without L1+L2 it lacked enough regularisation to beat the full layer4 runs on test F2.

This iteration combines both: **narrow unfreeze scope** (4.7M params, less capacity to overfit) **+ L1+L2 penalties** (the regularisation that demonstrably helped in 05). The hypothesis is that this combination will keep the train/val gap tight while preserving the test F2 gains from the penalties.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd()] + list(Path.cwd().parents) if (p / 'src').exists())
sys.path.insert(0, str(ROOT))

import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

from src.data.dataset import HAM10000Dataset
from src.data.dataloader import get_dataloaders
from src.data.transform import get_augmented_train_transforms
from src.models.resnet import get_resnet50
from src.training.trainer import train_one_epoch, validate_one_epoch
from src.utils import plot_training_curves, find_best_threshold, evaluate_model

In [ ]:
import random
import numpy as np

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

In [ ]:
train_dataset = HAM10000Dataset(
    csv_path=str(ROOT / 'data_new/splits/train.csv'),
    image_dir=str(ROOT / 'data_new/images/train'),
    transform=get_augmented_train_transforms(image_size=224),
)
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    persistent_workers=True,
)

_, val_loader, test_loader = get_dataloaders(
    train_csv=str(ROOT / 'data_new/splits/train.csv'),
    val_csv=str(ROOT / 'data_new/splits/val.csv'),
    test_csv=str(ROOT / 'data_new/splits/test.csv'),
    image_dir=str(ROOT / 'data_new/images/train'),
    test_image_dir=str(ROOT / 'data_new/images/test'),
    batch_size=32,
    image_size=224,
    num_workers=4,
)

train_df     = pd.read_csv(ROOT / 'data_new/splits/train.csv')
num_melanoma = (train_df['label'] == 1).sum()
num_nevus    = (train_df['label'] == 0).sum()
pos_weight   = torch.tensor([num_nevus / num_melanoma], dtype=torch.float32).to(device)
print('Positive weight:', pos_weight)

In [ ]:
model = get_resnet50(num_classes=1, freeze_backbone=True, dropout=0.5).to(device)

# Only unfreeze the last bottleneck block of layer4
for param in model.layer4[2].parameters():
    param.requires_grad = True

optimizer = optim.AdamW([
    {'params': model.layer4[2].parameters(), 'lr': 1e-4},
    {'params': model.fc.parameters(),        'lr': 1e-3},
], weight_decay=1e-3)

criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
num_epochs = 30
scheduler  = CosineAnnealingLR(optimizer, T_max=num_epochs)

L1_LAMBDA = 1e-4
L2_LAMBDA = 1e-4

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,}')
print(f'L1 lambda: {L1_LAMBDA} | L2 lambda: {L2_LAMBDA}')

In [ ]:
best_val_f2 = 0.0
train_history, val_history = [], []

for epoch in range(num_epochs):
    train_metrics = train_one_epoch(
        model, train_loader, criterion, optimizer, device,
        l1_lambda=L1_LAMBDA, l2_lambda=L2_LAMBDA,
    )
    val_metrics = validate_one_epoch(model, val_loader, criterion, device)

    scheduler.step()

    train_history.append(train_metrics)
    val_history.append(val_metrics)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] | "
        f"Train Loss: {train_metrics['loss']:.4f}, Bal Acc: {train_metrics['balanced_accuracy']:.4f}, "
        f"Recall: {train_metrics['recall']:.4f}, F2: {train_metrics['f2']:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f}, Bal Acc: {val_metrics['balanced_accuracy']:.4f}, "
        f"Recall: {val_metrics['recall']:.4f}, F2: {val_metrics['f2']:.4f}"
    )

    if val_metrics['f2'] > best_val_f2:
        best_val_f2 = val_metrics['f2']
        torch.save(model.state_dict(), ROOT / 'models/resnet50_layer4_2_l1_l2_best.pth')
        print(f'  -> Saved best model (val F2: {best_val_f2:.4f})')

## Training Curves

In [ ]:
plot_training_curves(train_history, val_history)

## Threshold Tuning

In [ ]:
model.load_state_dict(torch.load(str(ROOT / 'models/resnet50_layer4_2_l1_l2_best.pth'), map_location=device))
best_threshold, best_f2 = find_best_threshold(model, val_loader, device)

## Test Set Evaluation

In [ ]:
evaluate_model(model, test_loader, device, threshold=best_threshold)